# Read subjects and stories for secondlevel analysis

In [1]:
import pandas as pd
import os
import shutil
from subprocess import run

/home/xy6/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
mastersheetFile = '/work/desai-lab/xuanyang/Project/Semantic/dissemination/github/DiscoFMRI/scripts/fMRI/master_subject_10stories_highacc.csv'
df_master = pd.read_csv(mastersheetFile)
df_master.loc[df_master['transcript'].isin(['prettymouth','milkywayoriginal','milkywayvodka','slumlordreach','21styear']),'protocol'] = 'skyra'
df_master.loc[df_master['transcript'].isin(['shapessocial']),'protocol'] = 'Prisma_MB4'
df_master.loc[df_master['transcript'].isin(['piemanpni','bronx','black','forgot']),'protocol'] = 'Prisma_MB3'
df_master

,subID,task,age,sex,condition,comprehension,transcript,label,protocol
0,sub-023,prettymouth,28,F,affair,0.889,prettymouth,prettymouth,skyra
1,sub-023,milkyway,28,F,vodka,1.000,milkywayvodka,milkywayvodka,skyra
2,sub-030,prettymouth,21,F,paranoia,0.963,prettymouth,prettymouth,skyra
3,sub-030,milkyway,21,F,vodka,0.893,milkywayvodka,milkywayvodka,skyra
4,sub-032,prettymouth,22,M,affair,0.963,prettymouth,prettymouth,skyra
...,...,...,...,...,...,...,...,...,...
208,sub-314,piemanpni,25,F,NaN,0.767,piemanpni,piemanpni,Prisma_MB3
209,sub-314,bronx,25,F,NaN,0.780,bronx,bronx,Prisma_MB3
210,sub-314,forgot,25,F,NaN,0.780,forgot,forgot,Prisma_MB3
211,sub-314,black,25,F,NaN,0.920,black,black,Prisma_MB3


In [3]:
import subprocess
import pandas as pd

def get_subbricks_info(dataset):
    # Run 3dinfo command
    result = subprocess.run(['/work/apps/AFNI/linux_openmp_64/3dinfo', '-verb', dataset], capture_output=True, text=True)
    lines = result.stdout.split('\n')
    
    # Extract lines with sub-brick info
    subbrick_lines = [line for line in lines if "sub-brick" in line]
    
    # Extract indices and names from those lines
    subbricks = []
    for line in subbrick_lines:
        idx = int(line.split('#')[1].split()[0])
        name = line.split("'")[1]
        subbricks.append((idx, name))
        
    return subbricks


# Mask

In [4]:
Dir_working = '/work/desai-lab/xuanyang/Project/Semantic/analysis/ParametricModulation/Nastase/allstories/FactorAnalysis'
Nvar = 113
NFA = 8
flag_model = f'Nvar{Nvar}NFA{NFA}_LPAC_multipleReg'

Dir_figures = f'/work/desai-lab/xuanyang/Project/Semantic/analysis/ParametricModulation/Nastase/allstories/FactorAnalysis/models/{flag_model}/figures_SVC_LMEr/'
os.makedirs(Dir_figures,exist_ok=True)

# groupMask = '/work/desai-lab/xuanyang/Project/dataset/Nastase/narratives/derivatives/afni-nosmooth/tpl-MNI152NLin2009cAsym/nosmooth_mask_allstories.nii'
groupMask = '/work/desai-lab/xuanyang/Project/Semantic/analysis/FactorAnalysis/github/FactorAnalysis_fMRI/scripts/fMRI/masks/rs_mask_GM_33.nii.gz'
folder = 'threshold_GM_LMEr'



In [5]:
Dir_model = os.path.join(Dir_working,'models',flag_model,f'FA{NFA}')
Dir_results = os.path.join(Dir_model,'results')
Dir_results_1st = os.path.join(Dir_results,'firstlevel')
Dir_results_2nd = os.path.join(Dir_results,'secondlevel')

for i,subID, task,label in zip(df_master.index,df_master.subID,df_master.task,df_master.label):
    idx = "{:03d}".format(i+1)
    Dir_sub_task = os.path.join(Dir_results_1st,subID,task)
    path_img_stats = os.path.join(Dir_sub_task,'{}.results'.format(subID),'stats.{}_REML+tlrc.BRIK'.format(subID))
    df_master.loc[i,'finished'] = 0

    if os.path.exists(path_img_stats):
        df_master.loc[i,'finished'] = 1
        df_master.loc[i,'img_stats'] = path_img_stats

subfolder = '{}highacc'.format(len(df_master))
subList_ana_cmplt = df_master.loc[df_master.finished==1].reset_index(drop=True)
Nsub = len(df_master)

df_subbricks = pd.DataFrame(get_subbricks_info(subList_ana_cmplt.img_stats[0]),columns=['imgIDList', 'regName'])
df_regModel = df_subbricks.loc[df_subbricks.regName.str.contains(f'_bin2345#0_Coef')].copy()
df_regModel['regName'] = df_regModel['regName'].str.replace('_bin2345#0_Coef',f'_{flag_model}')
df_regModel['regName'] = df_regModel['regName'] +'_'+subfolder
df_regModel

,imgIDList,regName
114,114,FA1_Nvar113NFA8_LPAC_multipleReg_213highacc
117,117,FA2_Nvar113NFA8_LPAC_multipleReg_213highacc
120,120,FA3_Nvar113NFA8_LPAC_multipleReg_213highacc
123,123,FA4_Nvar113NFA8_LPAC_multipleReg_213highacc
126,126,FA5_Nvar113NFA8_LPAC_multipleReg_213highacc
129,129,FA6_Nvar113NFA8_LPAC_multipleReg_213highacc
132,132,FA7_Nvar113NFA8_LPAC_multipleReg_213highacc
135,135,FA8_Nvar113NFA8_LPAC_multipleReg_213highacc


In [6]:
Dir_model = os.path.join(Dir_working,'models',flag_model,f'FA{NFA}')
Dir_results = os.path.join(Dir_model,'results')
Dir_results_1st = os.path.join(Dir_results,'firstlevel')
Dir_results_2nd = os.path.join(Dir_results,'secondlevel')

for i,subID, task,label in zip(df_master.index,df_master.subID,df_master.task,df_master.label):
    idx = "{:03d}".format(i+1)
    Dir_sub_task = os.path.join(Dir_results_1st,subID,task)
    path_img_stats = os.path.join(Dir_sub_task,'{}.results'.format(subID),'stats.{}_REML+tlrc.BRIK'.format(subID))
    df_master.loc[i,'finished'] = 0

    if os.path.exists(path_img_stats):
        df_master.loc[i,'finished'] = 1
        df_master.loc[i,'img_stats'] = path_img_stats

subfolder = '{}highacc'.format(len(df_master))
subList_ana_cmplt = df_master.loc[df_master.finished==1].reset_index(drop=True)
Nsub = len(df_master)

df_subbricks = pd.DataFrame(get_subbricks_info(subList_ana_cmplt.img_stats[0]),columns=['imgIDList', 'regName'])
df_regModel = df_subbricks.loc[df_subbricks.regName.str.contains(f'_bin2345#0_Coef')].copy()
df_regModel['regName'] = df_regModel['regName'].str.replace('_bin2345#0_Coef',f'_{flag_model}')
df_regModel['regName'] =df_regModel['regName'] +'_'+subfolder

for regName,imgID in zip(df_regModel['regName'],df_regModel['imgIDList']):
    resultDir_2nd = os.path.join(Dir_results_2nd,folder)
    df_lme_table = df_master[['subID','protocol','transcript']].copy()
    df_lme_table.rename({'subID':'Subj'},axis=1,inplace=True)
    df_lme_table['InputFile'] = [f"{statImg.replace('.BRIK','')}[{imgID}]" for statImg in df_master['img_stats']]
    df_lme_table.to_csv(os.path.join(resultDir_2nd,f"dataTable_{regName}.txt"),index=False,sep='\t')

    
    masterScript = os.path.join(resultDir_2nd,'run_secondlevel_{}.sh'.format(subfolder))
    if not os.path.exists(resultDir_2nd):
        os.makedirs(resultDir_2nd)
    os.chdir(resultDir_2nd)
    scriptFile = os.path.join(resultDir_2nd,'Batch_secondlevel_'+regName)
    with open(scriptFile, "w") as f:
        f.write("#!/bin/tcsh -xef \n")
        f.write(f"/work/apps/AFNI/26.0.08/3dLMEr -prefix {regName} \\\n")
        f.write(f"-resid {regName}_resid \\\n")
        f.write(f"-mask {groupMask} \\\n")
        f.write("-model 'protocol+(1|Subj)+(1|transcript)' \\\n")
        f.write("-IF InputFile \\\n")
        f.write("-SS_type 3 \\\n")
        f.write("-gltCode mean 'protocol : 0.333333*skyra +0.333333*Prisma_MB3 +0.333333*Prisma_MB4' \\\n")
        f.write(f"-dataTable @/{resultDir_2nd}/dataTable_{regName}.txt")
#     os.chdir(resultDir_2nd)
#     run(f"tcsh {scriptFile}",shell=True)


# List_files = [x for x in os.listdir(resultDir_2nd) if '.nii' in x or '+tlrc' in x]
# for file in List_files:
#     os.makedirs(os.path.join(Dir_figures,folder),exist_ok=True)
#     shutil.copy(os.path.join(resultDir_2nd,file),os.path.join(Dir_figures,folder,file))
    

# Run slurm job

/work/desai-lab/xuanyang/Project/Semantic/analysis/ParametricModulation/Nastase/allstories/FactorAnalysis/models/Nvar113NFA8_LPAC_multipleReg/FA8/results/secondlevel/threshold_GM_LMEr/slurm_lmer.sh

# Estimate ACF

In [ ]:
mask=/work/desai-lab/xuanyang/Project/Semantic/analysis/FactorAnalysis/github/FactorAnalysis_fMRI/scripts/fMRI/masks/rs_mask_GM_33.nii.gz
for factor in {1..8}; do
    prefix="FA${factor}_Nvar113NFA8_LPAC_multipleReg_213highacc"

    3dFWHMx \
        -mask "$mask" \
        -ACF NULL \
        -input "${prefix}_resid+tlrc" \
        > "${prefix}_ACF.txt"
done

# 3dClustSim

In [11]:
# Configure and parse the residual ACF estimates produced by 3dFWHMx.
# ACF parameters describe the spatial correlation of the model residuals.
from pathlib import Path
import math
import re
import subprocess

clustsim_dir = Path(Dir_results_2nd) / folder
clustsim_dir.mkdir(parents=True, exist_ok=True)
afni_bin = Path('/work/apps/AFNI/26.0.08')
voxel_p = 0.001       # total two-sided voxelwise p-value
cluster_alpha = 0.05  # cluster-level FWE within each factor map
NN = 2                # face + edge neighbors
number_pattern = re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[Ee][-+]?\d+)?')

def numeric_values(text):
    return [float(value) for value in number_pattern.findall(text)]

def read_acf(acf_file):
    candidates = []
    for raw_line in Path(acf_file).read_text(errors='replace').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        values = numeric_values(line)
        if len(values) >= 3:
            a, b, c = values[:3]
            if 0 <= a <= 1 and b > 0 and c > 0:
                candidates.append((a, b, c))
    if not candidates:
        raise ValueError(f'No valid ACF triplet found in {acf_file}')
    return candidates[-1]

def read_cluster_cutoff(table_file, target_p=0.001, target_alpha=0.05):
    lines = Path(table_file).read_text(errors='replace').splitlines()
    alpha_values = None
    for index, line in enumerate(lines):
        if 'pthr' in line.lower() and 'alpha' in line.lower():
            for candidate in lines[index:index + 4]:
                if '|' in candidate:
                    values = numeric_values(candidate.split('|', 1)[1])
                    if values:
                        alpha_values = values
                        break
            if alpha_values:
                break
    if not alpha_values:
        raise ValueError(f'Could not read alpha columns from {table_file}')
    alpha_index = min(range(len(alpha_values)), key=lambda i: abs(alpha_values[i] - target_alpha))
    if not math.isclose(alpha_values[alpha_index], target_alpha, abs_tol=5e-7):
        raise ValueError(f'alpha={target_alpha} is absent from {table_file}')
    rows = []
    for line in lines:
        stripped = line.strip()
        if not stripped or stripped.startswith('#'):
            continue
        values = numeric_values(stripped)
        if len(values) >= 1 + len(alpha_values):
            rows.append((values[0], values[1:1 + len(alpha_values)]))
    if not rows:
        raise ValueError(f'Could not read p-threshold rows from {table_file}')
    row_p, cutoffs = min(rows, key=lambda row: abs(row[0] - target_p))
    if not math.isclose(row_p, target_p, abs_tol=5e-7):
        raise ValueError(f'p={target_p} is absent from {table_file}')
    raw_cutoff = cutoffs[alpha_index]
    return raw_cutoff, math.ceil(raw_cutoff)


In [12]:
# Generate and run one auditable 3dClustSim Bash script for each factor.
# Existing simulation tables are reused so an accidental notebook rerun is inexpensive.
clustsim_rows = []
for iFA in range(1, NFA + 1):
    prefix = f'FA{iFA}_{flag_model}_{subfolder}'
    acf_file = clustsim_dir / f'{prefix}_ACF.txt'
    if not acf_file.exists():
        raise FileNotFoundError(f'Missing ACF output: {acf_file}')

    acf_a, acf_b, acf_c = read_acf(acf_file)
    sim_prefix = clustsim_dir / f'{prefix}.CSimA'
    sim_table = Path(f'{sim_prefix}.NN{NN}_bisided.1D')
    bash_file = clustsim_dir / f'Batch_3dClustSim_{prefix}.sh'
    bash_text = f'''#!/usr/bin/env bash
set -euo pipefail
cd {clustsim_dir}
{afni_bin / '3dClustSim'} \
    -mask {groupMask} \
    -acf {acf_a:.10g} {acf_b:.10g} {acf_c:.10g} \
    -pthr {voxel_p:.10g} \
    -athr {cluster_alpha:.10g} \
    -LOTS \
    -prefix {sim_prefix}
'''
    bash_file.write_text(bash_text)
    bash_file.chmod(0o755)

    if not sim_table.exists():
        print(f'Running {bash_file.name}')
        subprocess.run(['bash', str(bash_file)], cwd=clustsim_dir, check=True)
    else:
        print(f'Reusing {sim_table.name}')

    raw_nvox, applied_nvox = read_cluster_cutoff(sim_table, voxel_p, cluster_alpha)
    clustsim_rows.append({
        'FA': iFA, 'acf_a': acf_a, 'acf_b': acf_b, 'acf_c': acf_c,
        'voxel_p_bisided': voxel_p, 'cluster_alpha': cluster_alpha, 'NN': NN,
        'raw_cluster_nvox': raw_nvox, 'applied_cluster_nvox': applied_nvox,
        'table': sim_table.name,
    })

df_clustsim = pd.DataFrame(clustsim_rows)
df_clustsim.to_csv(clustsim_dir / 'LMEr_3dClustSim_thresholds.csv', index=False)
df_clustsim


Reusing FA1_Nvar113NFA8_LPAC_multipleReg_213highacc.CSimA.NN2_bisided.1D
Running Batch_3dClustSim_FA2_Nvar113NFA8_LPAC_multipleReg_213highacc.sh


++ 3dClustSim: AFNI version=AFNI_26.0.08 (Jan 30 2026) [64-bit]
++ Authored by: RW Cox and BD Ward
++ 41229 voxels in mask (16.81% of total)
++ Kernel function radius = 25.88 mm
++ ACF(0.57,2.84,8.47) => FWHM=7.69 => 65x77x49 pads to 96x96x64
 + Kernel image dimensions 47 x 47 x 31
++ Startup clock time = 0.1 s
++ Using 15 OpenMP threads
Simulating:0123456789.0123456789.0123456789.0123456789.01234567!
++ Clock time now = 137.1 s


Running Batch_3dClustSim_FA3_Nvar113NFA8_LPAC_multipleReg_213highacc.sh


++ 3dClustSim: AFNI version=AFNI_26.0.08 (Jan 30 2026) [64-bit]
++ Authored by: RW Cox and BD Ward
++ 41229 voxels in mask (16.81% of total)
++ Kernel function radius = 26.61 mm
++ ACF(0.59,2.84,8.79) => FWHM=7.70 => 65x77x49 pads to 96x96x64
 + Kernel image dimensions 47 x 47 x 31
++ Startup clock time = 0.1 s
++ Using 15 OpenMP threads
Simulating:0123456789.0123456789.0123456789.0123456789.01234567!
++ Clock time now = 158.8 s


Running Batch_3dClustSim_FA4_Nvar113NFA8_LPAC_multipleReg_213highacc.sh


...

Running Batch_3dClustSim_FA7_Nvar113NFA8_LPAC_multipleReg_213highacc.sh


++ 3dClustSim: AFNI version=AFNI_26.0.08 (Jan 30 2026) [64-bit]
++ Authored by: RW Cox and BD Ward
++ 41229 voxels in mask (16.81% of total)
++ Kernel function radius = 26.17 mm
++ ACF(0.57,2.85,8.56) => FWHM=7.73 => 65x77x49 pads to 96x96x64
 + Kernel image dimensions 47 x 47 x 31
++ Startup clock time = 0.1 s
++ Using 15 OpenMP threads
Simulating:0123456789.0123456789.0123456789.0123456789.01234567!
++ Clock time now = 146.3 s


Running Batch_3dClustSim_FA8_Nvar113NFA8_LPAC_multipleReg_213highacc.sh


++ 3dClustSim: AFNI version=AFNI_26.0.08 (Jan 30 2026) [64-bit]
++ Authored by: RW Cox and BD Ward
++ 41229 voxels in mask (16.81% of total)
++ Kernel function radius = 25.69 mm
++ ACF(0.57,2.84,8.40) => FWHM=7.68 => 65x77x49 pads to 96x96x64
 + Kernel image dimensions 47 x 47 x 31
++ Startup clock time = 0.1 s
++ Using 15 OpenMP threads
Simulating:0123456789.0123456789.0123456789.0123456789.01234567!
++ Clock time now = 149.1 s


,FA,acf_a,acf_b,acf_c,voxel_p_bisided,cluster_alpha,NN,raw_cluster_nvox,applied_cluster_nvox,table
0,1,0.576561,2.84001,8.61379,0.001,0.05,2,8.1,9,FA1_Nvar113NFA8_LPAC_multipleReg_213highacc.CS...
1,2,0.574893,2.84217,8.46724,0.001,0.05,2,7.0,7,FA2_Nvar113NFA8_LPAC_multipleReg_213highacc.CS...
2,3,0.586963,2.83691,8.78774,0.001,0.05,2,8.1,9,FA3_Nvar113NFA8_LPAC_multipleReg_213highacc.CS...
3,4,0.575552,2.84447,9.01126,0.001,0.05,2,8.1,9,FA4_Nvar113NFA8_LPAC_multipleReg_213highacc.CS...
4,5,0.576137,2.84920,8.81443,0.001,0.05,2,7.4,8,FA5_Nvar113NFA8_LPAC_multipleReg_213highacc.CS...
5,6,0.578035,2.84836,8.46206,0.001,0.05,2,7.4,8,FA6_Nvar113NFA8_LPAC_multipleReg_213highacc.CS...
6,7,0.574111,2.85221,8.55616,0.001,0.05,2,8.1,9,FA7_Nvar113NFA8_LPAC_multipleReg_213highacc.CS...
7,8,0.574473,2.84057,8.40062,0.001,0.05,2,7.9,8,FA8_Nvar113NFA8_LPAC_multipleReg_213highacc.CS...


In [41]:
subfolder = '{}highacc'.format(len(df_master))
# os.makedirs(os.path.join(Dir_results_2nd,folder),exist_ok=True)
os.chdir(os.path.join(Dir_results_2nd,folder))
for iFA in range(1,NFA+1):#NFA+1
    file = f'FA{iFA}_{flag_model}_{subfolder}'
    print(file)
    script = f'#!/bin/tcsh -xef \n /work/apps/AFNI/linux_openmp_64/3dAFNItoNIFTI {file}+tlrc.BRIK"[1]" -prefix {file}.nii'
    run(f"tcsh {script}",shell=True)
    
    script = f'#!/bin/tcsh -xef \n /work/apps/AFNI/linux_openmp_64/3dcalc \
        -a {file}+tlrc.BRIK"[2]" \
        -expr "fizt_t2p(a) * step(a) - fizt_t2p(a) * step(-a)" \
        -prefix {file}_p.nii \
        -datum float'
    run(f"tcsh {script}",shell=True)

FA1_Nvar113NFA8_LPAC_multipleReg_213highacc


++ 3dAFNItoNIFTI: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ 3dcalc: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ Authored by: A cast of thousands
++ Output dataset ./FA1_Nvar113NFA8_LPAC_multipleReg_213highacc_p.nii
++ 3dAFNItoNIFTI: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]


FA2_Nvar113NFA8_LPAC_multipleReg_213highacc


++ 3dcalc: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ Authored by: A cast of thousands
++ Output dataset ./FA2_Nvar113NFA8_LPAC_multipleReg_213highacc_p.nii
++ 3dAFNItoNIFTI: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]


FA3_Nvar113NFA8_LPAC_multipleReg_213highacc


...

++ 3dAFNItoNIFTI: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ 3dcalc: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ Authored by: A cast of thousands
++ Output dataset ./FA6_Nvar113NFA8_LPAC_multipleReg_213highacc_p.nii
++ 3dAFNItoNIFTI: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ 3dcalc: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ Authored by: A cast of thousands


FA7_Nvar113NFA8_LPAC_multipleReg_213highacc


++ Output dataset ./FA7_Nvar113NFA8_LPAC_multipleReg_213highacc_p.nii
++ 3dAFNItoNIFTI: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ 3dcalc: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ Authored by: A cast of thousands


FA8_Nvar113NFA8_LPAC_multipleReg_213highacc


++ Output dataset ./FA8_Nvar113NFA8_LPAC_multipleReg_213highacc_p.nii


In [16]:
for iFA in range(1,NFA+1):#NFA+1
    Dir_model = os.path.join(Dir_working,'models',flag_model,f'FA{NFA}')
    Dir_results = os.path.join(Dir_model,'results','secondlevel',folder)
    file = f"FA{iFA}_{flag_model}_{subfolder}.CSimA.NN2_bisided.1D"
    file_src = os.path.join(Dir_results,file)
    os.makedirs(os.path.join(Dir_figures,folder),exist_ok=True)
    file_dst = os.path.join(Dir_figures,folder,file)

    shutil.copy(file_src,file_dst)
    

# Clusterize threshold

In [43]:
NN = 2
sided = 'bisided'
os.chdir(os.path.join(Dir_figures,folder))

alpha = '0.05'
for iFA in range(1,NFA+1):#NFA+1
    file_table = os.path.join(Dir_figures,folder,f"FA{iFA}_{flag_model}_{subfolder}.CSimA.NN{NN}_{sided}.1D")
    df = pd.read_csv(file_table, delim_whitespace=True, skiprows=8)
    new_row = pd.DataFrame([df.columns], columns=df.columns)
    df = pd.concat([new_row, df], ignore_index=True)
    df.columns = ['p']+[f"0.{11-x:02d}" for x in range(1,11)]
    
    for p in [0.001,0.005]: 
        if sided != '1sided':
            p = p*2 
            threshold_label = f"{sided}NN{NN}_1tailp{str(p/2)[2:]}"
        else:
            threshold_label = f"{sided}NN{NN}_1tailp{str(p)[2:]}"
            
            
        voxels = df.loc[df['p']==p,alpha].values[0]
        
        Dir_output = os.path.join(Dir_figures,folder,threshold_label)
        if not os.path.exists(Dir_output):
            os.makedirs(Dir_output)
                
        file_input = os.path.join(Dir_results_2nd,folder,f'FA{iFA}_{flag_model}_{subfolder}+tlrc')
        file_output_dat = os.path.join(Dir_output,f'FA{iFA}_{flag_model}_{subfolder}_v{voxels}_dat')
        file_output_map = os.path.join(Dir_output,f'FA{iFA}_{flag_model}_{subfolder}_v{voxels}_ROI')
        file_output_1D = os.path.join(Dir_output,f'FA{iFA}_{flag_model}_{subfolder}_v{voxels}_ROI.1D')
        script = f"3dClusterize -inset {file_input} -mask {groupMask} -NN {NN} -ithr 2 -idat 1 -clust_nvox {voxels} -pref_dat {file_output_dat} -pref_map {file_output_map} -{sided} p={p} -1Dformat > {file_output_1D}"
        print(script)
        print('\n')
    #     script = f"#!/bin/tcsh -xef \n /work/apps/AFNI/linux_openmp_64/3dClusterize -inset {file_input} -NN {NN} -ithr 0 -clust_nvox {voxels} -pref_dat {file_output_dat} -thr_p {p}"
#         run(f"tcsh {script}",shell=True)




3dClusterize -inset /work/desai-lab/xuanyang/Project/Semantic/analysis/ParametricModulation/Nastase/allstories/FactorAnalysis/models/Nvar113NFA8_LPAC_multipleReg/FA8/results/secondlevel/threshold_GM_LMEr/FA1_Nvar113NFA8_LPAC_multipleReg_213highacc+tlrc -mask /work/desai-lab/xuanyang/Project/Semantic/analysis/FactorAnalysis/github/FactorAnalysis_fMRI/scripts/fMRI/masks/rs_mask_GM_33.nii.gz -NN 2 -ithr 2 -idat 1 -clust_nvox 11.3 -pref_dat /work/desai-lab/xuanyang/Project/Semantic/analysis/ParametricModulation/Nastase/allstories/FactorAnalysis/models/Nvar113NFA8_LPAC_multipleReg/figures_SVC_LMEr/threshold_GM_LMEr/bisidedNN2_1tailp001/FA1_Nvar113NFA8_LPAC_multipleReg_213highacc_v11.3_dat -pref_map /work/desai-lab/xuanyang/Project/Semantic/analysis/ParametricModulation/Nastase/allstories/FactorAnalysis/models/Nvar113NFA8_LPAC_multipleReg/figures_SVC_LMEr/threshold_GM_LMEr/bisidedNN2_1tailp001/FA1_Nvar113NFA8_LPAC_multipleReg_213highacc_v11.3_ROI -bisided p=0.002 -1Dformat > /work/desai-lab/

/work/xy6/tmp/ipykernel_41096/3127454753.py:8: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_table, delim_whitespace=True, skiprows=8)
/work/xy6/tmp/ipykernel_41096/3127454753.py:8: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_table, delim_whitespace=True, skiprows=8)
/work/xy6/tmp/ipykernel_41096/3127454753.py:8: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_table, delim_whitespace=True, skiprows=8)
/work/xy6/tmp/ipykernel_41096/3127454753.py:8: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_table, delim_whitespac

In [49]:
df_fig = []
for iFA in range(1,NFA+1):#NFA+1
    file_table = os.path.join(Dir_figures,folder,f"FA{iFA}_{flag_model}_{subfolder}.CSimA.NN{NN}_{sided}.1D")
    df = pd.read_csv(file_table, delim_whitespace=True, skiprows=8)
    new_row = pd.DataFrame([df.columns], columns=df.columns)
    df = pd.concat([new_row, df], ignore_index=True)
    df.columns = ['p']+[f"0.{11-x:02d}" for x in range(1,11)]
    
    
    for p in [0.001,0.005]: 
        if sided != '1sided':
            p = p*2 
            threshold_label = f"{sided}NN{NN}_1tailp{str(p/2)[2:]}"
        else:
            threshold_label = f"{sided}NN{NN}_1tailp{str(p)[2:]}"
            
        
        voxels = df.loc[df['p']==p,alpha].values[0]
        
        Dir_output = os.path.join(Dir_figures,folder,threshold_label)
        
        os.chdir(Dir_output)

        file = f'FA{iFA}_{flag_model}_{subfolder}_v{voxels}_dat'
        # file_output_dat = os.path.join(Dir_output,file)
        script = f'#!/bin/tcsh -xef \n /work/apps/AFNI/linux_openmp_64/3dAFNItoNIFTI {file}+tlrc.BRIK"[0]" -prefix {file}.nii'
        run(f"tcsh {script}",shell=True)
        
        if not os.path.exists(os.path.join(Dir_output,f"{file.replace('_dat','_mask')}.nii")):
            command = f"/work/apps/AFNI/linux_openmp_64/3dcalc -a {file}.nii -expr 'step(abs(a))' -prefix {file.replace('_dat','_mask')}.nii"
            run(f"{command}",shell=True)

        # get z-map
        file_input = os.path.join(Dir_results_2nd,folder,f'FA{iFA}_{flag_model}_{subfolder}+tlrc')
        file_mask = os.path.join(Dir_output,f"{file.replace('_dat','_mask')}.nii")
        script = f'#!/bin/tcsh -xef \n /work/apps/AFNI/linux_openmp_64/3dcalc -a {file_input}.BRIK"[1]" -b {file_mask} -expr "a*b" -prefix {file}_z.nii'
        run(f"tcsh {script}",shell=True)

        # get p-map
        script = f'#!/bin/tcsh -xef \n /work/apps/AFNI/linux_openmp_64/3dcalc \
        -a {file}_z.nii \
        -expr "fizt_t2p(a) * step(a) - fizt_t2p(a) * step(-a)" \
        -prefix {file}_p.nii'
        run(f"tcsh {script}",shell=True)

        # get min max
        script = f"/work/apps/AFNI/linux_openmp_64/3dinfo -min -max {os.path.join(Dir_output,file)}.nii"
        result = run(f"{script}",shell=True,capture_output=True, text=True)
        Min, Max = result.stdout.replace('\n','').split('\t')
        
        script = f"/work/apps/AFNI/linux_openmp_64/3dinfo -min -max {os.path.join(Dir_output,file)}_z.nii"
        result = run(f"{script}",shell=True,capture_output=True, text=True)
        Min_z, Max_z = result.stdout.replace('\n','').split('\t')

        script = f"/work/apps/AFNI/linux_openmp_64/3dinfo -min -max {os.path.join(Dir_output,file)}_p.nii"
        result = run(f"{script}",shell=True,capture_output=True, text=True)
        Min_p, Max_p = result.stdout.replace('\n','').split('\t')
        
        df_tmp = pd.DataFrame({'FA':iFA,'sided':sided,'p':p,
                               'min':Min,'max':Max,
                               'min_z':Min_z,'max_z':Max_z,
                               'min_p':Min_p,'max_p':Max_p},index=[0])
        df_fig.append(df_tmp)
df_fig = pd.concat(df_fig,ignore_index=True)

df_fig

/work/xy6/tmp/ipykernel_41096/147787808.py:4: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_table, delim_whitespace=True, skiprows=8)
++ 3dAFNItoNIFTI: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ 3dcalc: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ Authored by: A cast of thousands
** ERROR: output dataset name 'FA1_Nvar113NFA8_LPAC_multipleReg_213highacc_v11.3_dat_z.nii' conflicts with existing file
** ERROR: dataset NOT written to disk!
++ 3dcalc: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ Authored by: A cast of thousands
** ERROR: output dataset name 'FA1_Nvar113NFA8_LPAC_multipleReg_213highacc_v11.3_dat_p.nii' conflicts with existing file
** ERROR: dataset NOT written to disk!
++ 3dAFNItoNIFTI: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ 3dcalc: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ Authored by: A cast of thousands
** E

++ 3dcalc: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ Authored by: A cast of thousands
** ERROR: output dataset name 'FA6_Nvar113NFA8_LPAC_multipleReg_213highacc_v26.4_dat_z.nii' conflicts with existing file
** ERROR: dataset NOT written to disk!
++ 3dcalc: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ Authored by: A cast of thousands
** ERROR: output dataset name 'FA6_Nvar113NFA8_LPAC_multipleReg_213highacc_v26.4_dat_p.nii' conflicts with existing file
** ERROR: dataset NOT written to disk!
/work/xy6/tmp/ipykernel_41096/147787808.py:4: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_table, delim_whitespace=True, skiprows=8)
++ 3dAFNItoNIFTI: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ 3dcalc: AFNI version=AFNI_21.1.01 (Apr  9 2021) [64-bit]
++ Authored by: A cast of thousands
** ERROR: output dataset name 'FA7_Nvar113NFA8_LPAC_multipleReg_213high

,FA,sided,p,min,max,min_z,max_z,min_p,max_p
0,1,bisided,0.002,-0.951655,0.656118,-0.951655,0.656118,-0.919469,0.882409
1,1,bisided,0.010,-0.951655,0.656118,-0.951655,0.656118,-0.933293,0.913015
2,2,bisided,0.002,-0.620019,0.83383,-0.620019,0.83383,-0.905585,0.877538
3,2,bisided,0.010,-0.620019,0.83383,-0.620019,0.83383,-0.925808,0.899858
4,3,bisided,0.002,-0.51854,0.342727,-0.51854,0.342727,-0.952169,0.928846
5,3,bisided,0.010,-0.51854,0.342727,-0.51854,0.342727,-0.952169,0.938803
6,4,bisided,0.002,-0.292187,0.475244,-0.292187,0.475244,-0.869173,0.923221
7,4,bisided,0.010,0,0.475244,0,0.475244,0,0.932213
8,5,bisided,0.002,0,0.327564,0,0.327564,0,0.900616
9,5,bisided,0.010,0,0.327564,0,0.327564,0,0.933173


# BrainNet

In [50]:
df_fig.loc[df_fig['p']==0.002,'min'].min(),df_fig.loc[df_fig['p']==0.002,'max'].max()

('-0.292187', '0.83383')

In [51]:
df_fig.loc[df_fig['p']==0.01,'min'].min(),df_fig.loc[df_fig['p']==0.01,'max'].max()

('-0.30959', '0.83383')

# Recode FA7 for visualization

In [63]:
import shutil
# backup
for p in [0.001,0.005]: 
    if sided != '1sided':
        p = p*2 
        threshold_label = f"{sided}NN{NN}_1tailp{str(p/2)[2:]}"
    else:
        threshold_label = f"{sided}NN{NN}_1tailp{str(p)[2:]}"

    List_files = [x for x in os.listdir(os.path.join(Dir_figures,folder,threshold_label)) if 'FA7' in x and '.nii' in x and 'resid' not in x]
    for file in List_files:
        file1 = os.path.join(Dir_figures,folder,threshold_label,file)
        file2 = os.path.join(Dir_figures,folder,threshold_label,file.replace('.nii','_raw.nii'))
        if not os.path.exists(file2):
            # shutil.copy(file1,file2)
            os.rename(file1,file2)

In [66]:
for p in [0.001,0.005]: 
    if sided != '1sided':
        p = p*2 
        threshold_label = f"{sided}NN{NN}_1tailp{str(p/2)[2:]}"
    else:
        threshold_label = f"{sided}NN{NN}_1tailp{str(p)[2:]}"


    List_files = [x for x in os.listdir(os.path.join(Dir_figures,folder,threshold_label)) if 'FA7' in x and '.nii' in x and 'resid' not in x]
    for file in List_files:
        script = f"/work/apps/AFNI/linux_openmp_64/3dcalc -a {os.path.join(Dir_figures,folder,threshold_label,file)} -expr '-a' -prefix {os.path.join(Dir_figures,folder,threshold_label,file.replace('_raw.nii','.nii'))}"
        result = run(f"{script}",shell=True,capture_output=True, text=True)
